In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataCleaning").getOrCreate()

df = spark.read.csv(
    "customers_data.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

In [ ]:
from pyspark.sql.functions import col, explode
from pyspark.sql.types import StructType, ArrayType

def flatten_df(df):
    
    def _flatten(schema, prefix=""):
        fields = []
        for field in schema.fields:
            name = f"{prefix}{field.name}"
            dtype = field.dataType
            if isinstance(dtype, StructType):
                # recurse into struct
                fields.extend(_flatten(dtype, prefix=name + "."))
            elif isinstance(dtype, ArrayType) and isinstance(dtype.elementType, StructType):
                # explode array of structs
                fields.append((name, "explode"))
            else:
                fields.append((name, "column"))
        return fields

    flat_fields = _flatten(df.schema)

    # Explode arrays first
    for f, t in flat_fields:
        if t == "explode":
            df = df.withColumn(f, explode(col(f)))

    # Build select expressions
    select_exprs = []
    for f, t in flat_fields:
        if t == "column":
            select_exprs.append(col(f).alias(f.replace(".", "_")))
        elif t == "explode":
            # flatten struct fields inside exploded array
            struct_fields = df.select(col(f + ".*")).schema.fields
            for sf in struct_fields:
                select_exprs.append(col(f + "." + sf.name).alias(f.replace(".", "_") + "_" + sf.name))

    return df.select(*select_exprs)


In [ ]:
nested_df = spark.read.json("nested_orders.json")

flat_df = flatten_df(nested_df)
flat_df.printSchema()
flat_df.show(truncate=False)


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkOptimization") \
    .getOrCreate()

customers = spark.read.csv(
    "customers_data.csv",
    header=True,
    inferSchema=True
)

products = spark.read.csv(
    "products_data.csv",
    header=True,
    inferSchema=True
)

orders = spark.read.csv(
    "orders_dataset.csv",
    header=True,
    inferSchema=True
)

skewed_df = spark.read.csv(
    "skewed_transactions.csv",
    header=True,
    inferSchema=True
)

In [ ]:
print(orders.rdd.getNumPartitions())

In [ ]:
orders_repart = orders.repartition(8)

print(orders_repart.rdd.getNumPartitions())

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import spark_partition_id

spark = SparkSession.builder.getOrCreate()

df = spark.read.csv(
    "skewed_transactions.csv",
    header=True,
    inferSchema=True
)
df.show()

In [ ]:
print(df.rdd.getNumPartitions())

In [ ]:
partition_distribution = df.withColumn(
    "partition_id",
    spark_partition_id()
)

partition_distribution.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show(truncate=False)

In [ ]:
repart_df = df.repartition(8)

In [ ]:
repart_df.withColumn(
    "partition_id",
    spark_partition_id()
).groupBy("partition_id") \
 .count() \
 .orderBy("partition_id") \
 .show()

In [ ]:
#coalesce()

df = spark.read.csv(
    "skewed_transactions.csv",
    header=True,
    inferSchema=True
)

df = df.repartition(10)

print(df.rdd.getNumPartitions())
#10 files will be created when we write it

In [ ]:
small_df = df.coalesce(3)

print(small_df.rdd.getNumPartitions())

In [ ]:
small_df.write.mode("overwrite").csv("few_files")

In [ ]:
#IMBALNCE
from pyspark.sql.functions import spark_partition_id

small_df.withColumn(
    "partition_id",
    spark_partition_id()
).groupBy("partition_id") \
 .count() \
 .show()

In [ ]:
#BROADCASTING

#BEFORE
joined = orders.join(products, "product_id")

joined.explain(True)

In [ ]:
from pyspark.sql.functions import broadcast

broadcast_joined = orders.join(
    broadcast(products),
    "product_id"
)

broadcast_joined.explain(True)

In [ ]:
#COMPARING NORMAL and BROADCAST
#ORDERS DATASET
from pyspark.sql import SparkSession
import random
import pandas as pd

spark = SparkSession.builder \
    .appName("BroadcastJoinDemo") \
    .getOrCreate()

orders = []

for i in range(1, 1000001):

    orders.append([
        i,
        random.randint(1, 500),      # product_id
        random.randint(1, 5),        # quantity
        random.randint(500, 10000)   # amount
    ])

orders_pdf = pd.DataFrame(
    orders,
    columns=[
        "order_id",
        "product_id",
        "quantity",
        "amount"
    ]
)

orders_df = spark.createDataFrame(orders_pdf)

In [ ]:
#PRODUCTS DATASET
products = []

categories = [
    "Electronics",
    "Fashion",
    "Books",
    "Home",
    "Sports"
]

for i in range(1, 501):

    products.append([
        i,
        f"Product_{i}",
        random.choice(categories),
        random.randint(100, 5000)
    ])

products_pdf = pd.DataFrame(
    products,
    columns=[
        "product_id",
        "product_name",
        "category",
        "price"
    ]
)

products_df = spark.createDataFrame(products_pdf)

In [ ]:
normal_join = orders_df.join(
    products_df,
    "product_id"
)
from pyspark.sql.functions import broadcast

broadcast_join = orders_df.join(
    broadcast(products_df),
    "product_id"
)

In [ ]:
import time

# NORMAL JOIN

start = time.time()
normal_join.count()
end = time.time()

print("Normal Join Time:", end - start)

# BROADCAST JOIN
start = time.time()
broadcast_join.count()
end = time.time()

print("Broadcast Join Time:", end - start)

#28.7 seconds -------------normal
#5.3 seconds ------------broadcast 

In [ ]:
#SALTING
skewed_df.groupBy("customer_id").count().show()

In [ ]:
joined = skewed_df.join(customers, "customer_id")
joined.show(100)

In [ ]:
before_partition_data = (
    skewed_df.rdd
    .mapPartitions(lambda x: [sum(1 for _ in x)])
    .collect()
)

print(before_partition_data)
[198656,45,49,65]

In [ ]:
#ADDING SALT COLUMN
from pyspark.sql.functions import floor, rand, concat_ws

salted_skewed = skewed_df.withColumn(
    "salt",
    floor(rand() * 5)
)

In [ ]:
#DUPLICATE SMALL TABLE
from pyspark.sql.functions import explode, array, lit

expanded_customers = customers.withColumn(
    "salt",
    explode(array(
        lit(0),
        lit(1),
        lit(2),
        lit(3),
        lit(4)
    ))
)

In [ ]:
salted_join = salted_skewed.join(
    expanded_customers,
    ["customer_id", "salt"]
)

In [ ]:
salted_join.show(10)

In [ ]:
#repartitioning based on salt
after_partition_data = (
    salted_join.repartition("customer_id", "salt")
    .rdd
    .mapPartitions(lambda x: [sum(1 for _ in x)])
    .collect()
)

print(after_partition_data)
[85,77,65]

In [ ]:
#CACHE

from pyspark.sql.functions import sum

sales_df = orders_df.groupBy("product_id") \
    .agg(
        sum("amount").alias("total_sales")
    )

In [ ]:
#WITHOUT CACHE
import time
start = time.time()
sales_df.show()
sales_df.count()
sales_df.orderBy("total_sales", ascending=False).show()
end = time.time()

print("Without Cache Time:", end - start)

In [ ]:
#apply cache
sales_df.cache()
sales_df.count()

In [ ]:
#RE-EXECUTING IT
start = time.time()
sales_df.show()
sales_df.count()
sales_df.orderBy("total_sales", ascending=False).show()
end = time.time()
print("With Cache Time:", end - start)

In [ ]:
print(sales_df.is_cached)

In [ ]:
sales_df.persist(StorageLevel.MEMORY_AND_DISK)

sales_df.count()

In [ ]:
#PERSIST   - remaining partitions stored on disk
from pyspark import StorageLevel
sales_df.persist(StorageLevel.MEMORY_ONLY)

sales_df.count()

In [ ]:
sales_df.persist(StorageLevel.MEMORY_AND_DISK)

sales_df.count()

In [ ]:
sales_df.persist(StorageLevel.DISK_ONLY)

sales_df.count()